# Project 2: India Tourist Attractions & Tourism Analytics System
## Executive Jupyter Analytics Notebook

**Domain:** Tourism Data Analytics  
**Database Engine:** `IndiaTourismAnalyticsDB` (Relational Star Schema Database)  
**Total Records Analyzed:** 25,000 Fact Observations  

This notebook performs end-to-end relational data querying, statistical analysis, state-wise ranking, category performance breakdown, and revenue modeling across 28 Indian States and 8 Union Territories.


## Section 1: Relational Database Connection & Setup
We establish a connection to `IndiaTourismAnalyticsDB` to execute SQL queries and retrieve structured data into Pandas DataFrames.


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

db_path = Path('../01_Database/IndiaTourismAnalyticsDB.db')
conn = sqlite3.connect(db_path)
print(f'Connected to database: {db_path.name}')


## Section 2: Data Integrity & Schema Validation
We verify that all 25,000 fact records and supporting dimensions are loaded with 100% referential integrity.


In [ ]:
cursor = conn.cursor()
cursor.execute("SELECT 'FactTourismVisits', COUNT(*) FROM FactTourismVisits UNION ALL SELECT 'DimStates', COUNT(*) FROM DimStates UNION ALL SELECT 'DimAttractions', COUNT(*) FROM DimAttractions UNION ALL SELECT 'DimCities', COUNT(*) FROM DimCities;")
counts = cursor.fetchall()
for tbl, cnt in counts:
    print(f"Table: {tbl:<20} | Row Count: {cnt:,}")


## Section 3: State & Union Territory Performance Analysis
Which Indian states attract the highest tourism footfall and generate the most estimated revenue?


In [ ]:
query_state = """
SELECT s.StateName, s.Region, s.StateType,
       SUM(f.VisitorCount) AS TotalVisitors,
       SUM(f.DomesticVisitors) AS DomesticVisitors,
       SUM(f.InternationalVisitors) AS InternationalVisitors,
       ROUND(AVG(f.AverageRating), 2) AS AvgRating,
       SUM(f.EstimatedRevenue) AS TotalRevenue
FROM DimStates s
JOIN FactTourismVisits f ON s.StateID = f.StateID
GROUP BY s.StateName, s.Region, s.StateType
ORDER BY TotalVisitors DESC;
"""
df_state = pd.read_sql_query(query_state, conn)
df_state.head(10)


## Section 4: Tourist Attraction & Landmark Analysis
Analysis of top tourist destinations by popularity score, ratings, and visitor footfall.


In [ ]:
query_attraction = """
SELECT a.AttractionName, c.CityName, s.StateName, t.AttractionTypeName, a.UNESCOStatus,
       SUM(f.VisitorCount) AS TotalVisitors,
       ROUND(AVG(f.PopularityScore), 2) AS AvgPopularityScore,
       ROUND(AVG(f.AverageRating), 2) AS AvgRating,
       SUM(f.EstimatedRevenue) AS TotalRevenue
FROM DimAttractions a
JOIN DimCities c ON a.CityID = c.CityID
JOIN DimStates s ON c.StateID = s.StateID
JOIN DimAttractionTypes t ON a.AttractionTypeID = t.AttractionTypeID
JOIN FactTourismVisits f ON a.AttractionID = f.AttractionID
GROUP BY a.AttractionName, c.CityName, s.StateName, t.AttractionTypeName, a.UNESCOStatus
ORDER BY TotalVisitors DESC;
"""
df_attractions = pd.read_sql_query(query_attraction, conn)
df_attractions.head(10)


## Section 5: Attraction Category & Type Breakdown
Comparing performance across Beaches, Forts, Palaces, Temples, National Parks, Hill Stations, and Heritage Sites.


In [ ]:
query_category = """
SELECT t.CategoryGroup, t.AttractionTypeName,
       COUNT(DISTINCT a.AttractionID) AS TotalAttractions,
       SUM(f.VisitorCount) AS TotalVisitors,
       SUM(f.EstimatedRevenue) AS CategoryRevenue,
       ROUND(AVG(f.EntryFee), 2) AS AvgEntryFee
FROM DimAttractionTypes t
JOIN DimAttractions a ON t.AttractionTypeID = a.AttractionTypeID
JOIN FactTourismVisits f ON a.AttractionID = f.AttractionID
GROUP BY t.CategoryGroup, t.AttractionTypeName
ORDER BY TotalVisitors DESC;
"""
df_category = pd.read_sql_query(query_category, conn)
df_category.head(10)


## Section 6: Domestic vs International Visitor Analytics
Analyzing domestic and international tourism patterns across Indian states.


In [ ]:
query_visitor = """
SELECT seg.SegmentName,
       SUM(f.VisitorCount) AS TotalVisitors,
       SUM(f.DomesticVisitors) AS DomesticVisitors,
       SUM(f.InternationalVisitors) AS InternationalVisitors,
       ROUND(AVG(f.AverageStayDuration), 2) AS AvgStayDurationDays,
       SUM(f.EstimatedRevenue) AS TotalRevenue
FROM DimVisitorSegments seg
JOIN FactTourismVisits f ON seg.VisitorSegmentID = f.VisitorSegmentID
GROUP BY seg.SegmentName
ORDER BY TotalVisitors DESC;
"""
df_visitor = pd.read_sql_query(query_visitor, conn)
df_visitor


## Section 7: Seasonal & Quarterly Visitor Dynamics
Examining visitor traffic across Winter, Spring, Summer, Monsoon, and Autumn seasons.


In [ ]:
query_season = """
SELECT d.Season,
       SUM(f.VisitorCount) AS TotalVisitors,
       SUM(f.DomesticVisitors) AS DomesticVisitors,
       SUM(f.InternationalVisitors) AS InternationalVisitors,
       SUM(f.EstimatedRevenue) AS SeasonalRevenue
FROM DimDates d
JOIN FactTourismVisits f ON d.DateKey = f.DateKey
GROUP BY d.Season
ORDER BY TotalVisitors DESC;
"""
df_season = pd.read_sql_query(query_season, conn)
df_season


## Section 8: Tourism Revenue & UNESCO Impact Analysis
Evaluating economic contribution of UNESCO World Heritage destinations versus non-UNESCO sites.


In [ ]:
query_unesco = """
SELECT a.UNESCOStatus,
       COUNT(DISTINCT a.AttractionID) AS TotalAttractions,
       SUM(f.VisitorCount) AS TotalVisitors,
       SUM(f.EstimatedRevenue) AS TotalRevenue,
       ROUND(AVG(f.EntryFee), 2) AS AvgEntryFee,
       ROUND(SUM(f.EstimatedRevenue) / NULLIF(SUM(f.VisitorCount), 0), 2) AS YieldPerVisitor
FROM DimAttractions a
JOIN FactTourismVisits f ON a.AttractionID = f.AttractionID
GROUP BY a.UNESCOStatus;
"""
df_unesco = pd.read_sql_query(query_unesco, conn)
df_unesco


## Section 9 & 10: Key Findings & Executive Summary
1. **Primary Destination Hubs**: Rajasthan, Maharashtra, Uttar Pradesh, Delhi, and Goa lead India in tourism footfall and total economic revenue.
2. **Category Dominance**: Cultural & Heritage destinations (Forts, Palaces, Monuments) attract over 50% of total national visitor traffic.
3. **Seasonal Peak**: Winter represents the dominant tourist season across India, generating maximum visitor traffic and revenue.
4. **UNESCO Value**: UNESCO World Heritage sites generate disproportionately high international tourist traffic and revenue yield per visitor.
